### **Package**

In [ ]:
import subprocess
import sys
import os

print(sys.version)

for pkg in ["pymoo", "GPy", "autogluon.tabular"]:
    module_name = pkg.split('.')[0]
    try:
        __import__(pkg)
        print(f"{pkg} is already installed.")
    except ImportError:
        print(f"{pkg} is not installed. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"{pkg} installed.")


import warnings
warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/PhD 2026-1 New metrics')

from src.opt_problem import build_problem, Benchmark_Problem, EvaluatePreRealCallback, evaluate_pre_real
from src.survival import Survival_standard, Survival_dual_ranking
from src.data import generate_data
from src.metrics import get_metrics
from src.models import GPR_RBF, gpr_pred_mean_std
from src.uncertainty import coverage, find_alpha
from src.experiment import run_experiment
from src.other_functions import mean_std, print_gpr_params
from src.plotting import plot_z_score, plot_obj_2d, plot_y_true_pred

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.lhs import LHS
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt


### **Main**

###### 1. Initial settings

In [ ]:
# Problem: dtlz1-7, omnitest, bnh, truss2d, welded_beam
problem_name = 'dtlz1'
n_var = 10
n_obj = 2
problem = build_problem(
    problem_name = problem_name,
    n_var = n_var,
    n_obj = n_obj)
print(f"Problem name: {problem_name}")
print(f"Cons: {problem.n_constr}")
print(f"Var: {n_var}")
print(f"Obj: {n_obj}")

# EC algorithm: n_gen, pop_size
n_gen = 100
pop_size = 100


# Metrics
hv, igd_plus, obj_min, obj_max, ref_point = get_metrics(
    problem_name = problem_name,
    problem = problem,
    n_var=n_var,
    n_obj=n_obj)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

np.set_printoptions(precision=4, suppress=True)

# Data: LHS sampling with seed 42
random_seed = 42
sampling = LHS()
sample_size = 11*n_var-1
X_train, y_train, X_val, y_val, X_test, y_test = generate_data(
    problem=problem,
    sample_size=sample_size,
    sampling = sampling,
    train_seed=random_seed,
    val_size=100,
    test_size=100,
    test_seed=1)

y_train_f1 = y_train[:, 0]
y_train_f2 = y_train[:, 1]
print(f"Sampling X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

plot_obj_2d(y_train,
            xlim=(-100, 700),
            ylim=(-100, 700))


###### 2. Surrogate model training (normal)

In [ ]:
model_f1 = GPR_RBF()
model_f2 = GPR_RBF()

model_f1.fit(X_train, y_train_f1)
model_f2.fit(X_train, y_train_f2)
print_gpr_params(model_f1, model_f2)

# Predict
pred_mean, pred_std, mean_f1, std_f1, mean_f2, std_f2 = gpr_pred_mean_std(
    model_f1,
    model_f2,
    X_test,
    noiseless=True)
print(f"\nGPR(RBF) MSE: {mean_squared_error(y_test, pred_mean):.2e}\n")
plot_y_true_pred(y_test,pred_mean)

###### 2.1 Surrogate model training (high bias)

In [ ]:
lengthscale_weight = 10
model_f1_bias = GPR_RBF()
model_f2_bias = GPR_RBF()
model_f1_bias.fit_high_bias_variance(X_train, y_train_f1,
                                     lengthscale_weight=lengthscale_weight)
model_f2_bias.fit_high_bias_variance(X_train, y_train_f2,
                                     lengthscale_weight=lengthscale_weight)
print_gpr_params(model_f1_bias, model_f2_bias)

# Predict
pred_mean_bias, pred_std_bias, _,_,_,_= gpr_pred_mean_std(
    model_f1_bias,
    model_f2_bias,
    X_test,
    noiseless=True)
print(f"\nGPR(RBF) MSE: {mean_squared_error(y_test, pred_mean_bias):.2e}")
plot_y_true_pred(y_test,pred_mean_bias)

###### 2.2 Surrogate model training (high variance)

In [ ]:
lengthscale_weight = 0.1
model_f1_var = GPR_RBF()
model_f2_var = GPR_RBF()
model_f1_var.fit_high_bias_variance(X_train, y_train_f1,
                                     lengthscale_weight=lengthscale_weight)
model_f2_var.fit_high_bias_variance(X_train, y_train_f2,
                                     lengthscale_weight=lengthscale_weight)
print_gpr_params(model_f1_var, model_f2_var)

# Predict
pred_mean_variance, pred_std_variance, _,_,_,_= gpr_pred_mean_std(
    model_f1_var,
    model_f2_var,
    X_test,
    noiseless=True)
print(f"\nGPR(RBF) MSE: {mean_squared_error(y_test, pred_mean_variance):.2e}")
plot_y_true_pred(y_test,pred_mean_variance)

### Bias-Variance Experiment (30 samplings / 30 trainings)


In [ ]:
n_runs = 30

def run_bias_variance_experiment(lengthscale_weight, n_runs=30):
    preds = []

    for seed in range(1, n_runs + 1):
        X_train_i, y_train_i, _, _, X_test_i, y_test_i = generate_data(
            problem=problem,
            sample_size=sample_size,
            sampling=LHS(),
            train_seed=seed,
            val_size=100,
            test_size=100,
            test_seed=1)

        model_f1_i = GPR_RBF()
        model_f2_i = GPR_RBF()
        model_f1_i.fit_high_bias_variance(
            X_train_i, y_train_i[:, 0], lengthscale_weight=lengthscale_weight)
        model_f2_i.fit_high_bias_variance(
            X_train_i, y_train_i[:, 1], lengthscale_weight=lengthscale_weight)

        pred_mean_i, _, _, _, _, _ = gpr_pred_mean_std(
            model_f1_i, model_f2_i, X_test_i, noiseless=True)
        preds.append(pred_mean_i)

    preds = np.array(preds)  # (n_runs, n_test, n_obj)
    mean_pred = preds.mean(axis=0)
    bias = np.mean((mean_pred - y_test_i) ** 2)
    variance = np.mean(np.var(preds, axis=0))
    return bias, variance

high_bias_bias, high_bias_variance = run_bias_variance_experiment(
    lengthscale_weight=10, n_runs=n_runs)
high_var_bias, high_var_variance = run_bias_variance_experiment(
    lengthscale_weight=0.1, n_runs=n_runs)

print(f"High bias model -> Bias: {high_bias_bias:.6e}, Variance: {high_bias_variance:.6e}")
print(f"High variance model -> Bias: {high_var_bias:.6e}, Variance: {high_var_variance:.6e}")



###### CICP & Z score; find_alpha

In [ ]:
###### CICP
"""
Interval    Coverage
±1.282σ      80.00%
±1.645σ      90.00%
±1.960σ      95.00%
"""

for k in [1.645]:
    per_dim, overall = coverage(y_test, pred_mean, pred_std, k=k)
    print(f"k={k}: per_dim={per_dim*100}%, overall={overall*100:.1f}%")

###### Z score
plot_z_score(y_test, pred_mean, pred_std)

alpha_c90_f1 = find_alpha(
    X_val, y_val[:,0],
    model_f1,
    target_coverage=0.9)
alpha_c90_f2 = find_alpha(
    X_val, y_val[:,1],
    model_f2,
    target_coverage=0.9)
print(f"alpha_c90_f1={alpha_c90_f1:.2f}, alpha_c90_f2={alpha_c90_f2:.2f}")

F_upper_c90 = np.stack(
    [mean_f1 + alpha_c90_f1 * std_f1,
     mean_f2 + alpha_c90_f2 * std_f2], axis=1)
print('F_upper_c90\n', F_upper_c90[0:5],'\n')

###### 3. Optimization

In [ ]:
results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    model_f1=model_f1,
    model_f2=model_f2,
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_surrogate="GPR_uncertainty", ### change
    survival_function=Survival_standard(), ### change
    use_callback=False, ### change
    seeds=range(1, 31)) ### change

mse_list = results["mse_list"]
igd_list = results["igd_list"]
hv_surrogate_list = results["hv_surrogate_list"]
hv_real_list = results["hv_real_list"]

obj = results["run_details"][-1]["obj"]
f_real = results["run_details"][-1]["f_real"]
solution = results["run_details"][-1]["solution"]

plot_obj_2d(obj,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))


In [ ]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== GPR ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

###### 3. Optimization GPR (RBF) + dual-ranking (c=0.90)

In [ ]:
results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    model_f1=model_f1,
    model_f2=model_f2,
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_surrogate="GPR_uncertainty", ### change
    survival_function=Survival_dual_ranking(
        alpha_f1=alpha_c90_f1,
        alpha_f2=alpha_c90_f2), ### change
    use_callback=False, ### change
    seeds=range(1, 2)) ### change

mse_list = results["mse_list"]
igd_list = results["igd_list"]
hv_surrogate_list = results["hv_surrogate_list"]
hv_real_list = results["hv_real_list"]

obj = results["run_details"][-1]["obj"]
f_real = results["run_details"][-1]["f_real"]
solution = results["run_details"][-1]["solution"]

plot_obj_2d(obj,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== GPR + dual-ranking + alpha_c90  ===")

# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")